# RecruitBrain — Kaggle Precompute (BGE-M3, best quality)

**Before running:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Enable Internet: Settings → Internet → On
3. Add dataset: Add Data → Your datasets → redrob-candidates (candidates.jsonl)

**Output files — download and drop into `artifacts/` on your machine:**
- `candidate_embeddings.npy` (~400MB, BGE-M3 1024-dim)
- `faiss.index`
- `candidate_ids_kaggle.json`

**Expected runtime: ~8-12 min on T4 GPU**

In [ ]:
# Cell 1: Install
!pip install fastembed faiss-gpu numpy -q
# faiss-gpu uses GPU automatically when available
print('Install done')

In [ ]:
# Cell 2: Find candidates.jsonl
import os

# Kaggle mounts datasets under /kaggle/input/
CANDIDATES_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'candidates.jsonl':
            CANDIDATES_PATH = os.path.join(root, f)
            break

if not CANDIDATES_PATH:
    raise FileNotFoundError('candidates.jsonl not found — add the dataset in notebook settings')

print(f'Found: {CANDIDATES_PATH}')

# Count lines
with open(CANDIDATES_PATH, encoding='utf-8') as f:
    total = sum(1 for line in f if line.strip())
print(f'Total candidates: {total}')
assert total == 100000, f'Expected 100000, got {total}'

In [ ]:
# Cell 3: Build embedding texts
import json
import time

def build_text(c):
    """Rich text representation for embedding — same logic as src/candidate_loader.py"""
    prof = c['profile']
    parts = []

    # Title + headline + company — strong role signal
    parts.append(
        f"Title: {prof.get('current_title','')}. "
        f"{prof.get('headline','')}. "
        f"Company: {prof.get('current_company','')}. "
        f"Industry: {prof.get('current_industry','')}. "
        f"YOE: {prof.get('years_of_experience',0)}."
    )

    # Summary — richest free text
    summary = prof.get('summary', '')
    if summary:
        parts.append(summary)

    # Career descriptions — production evidence lives here
    for role in c.get('career_history', [])[:4]:
        desc = role.get('description', '')
        if desc:
            parts.append(
                f"Role at {role.get('company','')} as {role.get('title','')}: {desc[:400]}"
            )

    # Skills with proficiency
    skill_parts = [
        f"{s.get('name','')} ({s.get('proficiency','')})"
        for s in c.get('skills', [])[:20]
    ]
    if skill_parts:
        parts.append('Skills: ' + ', '.join(skill_parts))

    # Education
    for edu in c.get('education', [])[:2]:
        parts.append(
            f"Education: {edu.get('degree','')} in "
            f"{edu.get('field_of_study','')} from "
            f"{edu.get('institution','')} (tier: {edu.get('tier','')})"
        )

    return ' '.join(p for p in parts if p)


print('Loading candidates...')
t0 = time.time()
texts = []
ids = []

with open(CANDIDATES_PATH, encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            c = json.loads(line)
        except json.JSONDecodeError:
            continue
        texts.append(build_text(c))
        ids.append(c['candidate_id'])
        if (i + 1) % 10000 == 0:
            print(f'  {i+1} loaded...')

print(f'Total: {len(texts)} candidates in {time.time()-t0:.1f}s')
print(f'Sample text length: {len(texts[0])} chars')
print(f'Sample: {texts[0][:200]}')

In [ ]:
# Cell 4: Embed with BGE-M3 (SOTA 2024 retrieval model)
import numpy as np
from fastembed import TextEmbedding
import time

print('Loading BGE-M3 model (~580MB download)...')
t0 = time.time()

# BGE-M3: SOTA multilingual retrieval model (2024)
# 1024-dim, trained specifically for retrieval/ranking tasks
model = TextEmbedding(
    model_name='BAAI/bge-m3',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']  # GPU first, CPU fallback
)
print(f'Model loaded in {time.time()-t0:.1f}s')

print('\nEmbedding 100K candidates (~8-12 min on GPU)...')
t1 = time.time()

embeddings = np.array(
    list(model.embed(texts, batch_size=64)),
    dtype=np.float32
)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Embedding done in {time.time()-t1:.1f}s')

# L2 normalize for cosine similarity via inner product
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1.0, norms)
embeddings = embeddings / norms

print('Normalized.')

In [ ]:
# Cell 5: Save embeddings + build FAISS + save IDs
import faiss
import json
import numpy as np

# Save embeddings
np.save('/kaggle/working/candidate_embeddings.npy', embeddings)
print(f'Saved: candidate_embeddings.npy ({embeddings.nbytes / 1e6:.0f} MB)')

# Build FAISS index
print('Building FAISS index...')
dim = embeddings.shape[1]  # 1024 for BGE-M3

# Use GPU FAISS if available, else CPU
try:
    res = faiss.StandardGpuResources()
    index_cpu = faiss.IndexFlatIP(dim)
    index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
    index.add(embeddings)
    index = faiss.index_gpu_to_cpu(index)  # convert back to save
    print('Used GPU FAISS')
except Exception:
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print('Used CPU FAISS')

faiss.write_index(index, '/kaggle/working/faiss.index')
print(f'Saved: faiss.index ({index.ntotal} vectors, dim={dim})')

# Save ordered IDs
with open('/kaggle/working/candidate_ids_kaggle.json', 'w') as f:
    json.dump(ids, f)
print(f'Saved: candidate_ids_kaggle.json ({len(ids)} ids)')

In [ ]:
# Cell 6: Verify everything
import numpy as np
import faiss
import json

emb = np.load('/kaggle/working/candidate_embeddings.npy')
idx = faiss.read_index('/kaggle/working/faiss.index')
loaded_ids = json.load(open('/kaggle/working/candidate_ids_kaggle.json'))

print(f'Embeddings  : {emb.shape}')        # (100000, 1024)
print(f'FAISS       : {idx.ntotal} vectors') # 100000
print(f'IDs         : {len(loaded_ids)}')   # 100000

assert emb.shape[0] == idx.ntotal == len(loaded_ids), 'COUNT MISMATCH'
assert emb.shape[1] == 1024, f'Wrong dim: {emb.shape[1]} (expected 1024 for BGE-M3)'

# Quick sanity: query with JD text, top result should be an AI/ML engineer
jd_text = (
    "Senior AI Engineer founding team. Embeddings retrieval ranking vector databases FAISS "
    "Pinecone Weaviate hybrid search NLP LLM RAG evaluation NDCG MRR production ML Python."
)
query_emb = np.array(list(model.embed([jd_text])), dtype=np.float32)
query_emb = query_emb / np.linalg.norm(query_emb)
scores, indices = idx.search(query_emb.reshape(1, -1), 5)
print('\nTop 5 candidates for JD query:')
for rank, (score, idx_val) in enumerate(zip(scores[0], indices[0]), 1):
    print(f'  #{rank} {loaded_ids[idx_val]} score={score:.4f}')

print('\nALL OK')
print('Files ready to download:')
import os
for f in ['candidate_embeddings.npy', 'faiss.index', 'candidate_ids_kaggle.json']:
    size = os.path.getsize(f'/kaggle/working/{f}') / 1e6
    print(f'  /kaggle/working/{f}  ({size:.0f} MB)')

## Download files

After Cell 6 shows ALL OK:

1. In Kaggle sidebar → **Output** tab (folder icon on right)
2. Navigate to `/kaggle/working/`
3. Download these 3 files:
   - `candidate_embeddings.npy`
   - `faiss.index`
   - `candidate_ids_kaggle.json`
4. Drop all 3 into `artifacts/` on your local machine
5. Rename `candidate_ids_kaggle.json` → `candidate_ids.json` (replaces existing)

**Note:** `candidate_embeddings.npy` will be ~400MB (1024-dim × 100K × 4 bytes). Download may take a few minutes.

Once downloaded → we build `rank.py` and generate the submission CSV.